# Baselines

Two baselines against which QLoRA adapters are compared:
- **B1 -- Zero-shot:** Base model with artist name in prompt, no fine-tuning
- **B2 -- Few-shot:** Base model with artist name + 3 example lyrics in prompt

In [1]:
import json

import pandas as pd

from config import ARTISTS, RESULTS_DIR

# Display-only: baselines are computed + cached by evaluate.py.
# Run `uv run python evaluate.py` first.
BASELINES_DIR = RESULTS_DIR / "baselines"


def load_baseline(artist, method):
    f = BASELINES_DIR / artist.lower().replace(" ", "_") / f"{method}.json"
    if not f.exists():
        return None
    hit = json.load(open(f))
    return {"samples": hit["samples"], "df": pd.DataFrame(hit["df"])}

## B1 -- Zero-shot

Artist name in prompt, no examples, no fine-tuning.

In [2]:
zero_shot = {}
for artist in ARTISTS:
    hit = load_baseline(artist, "zero_shot")
    if hit is None:
        print(f"missing {artist} zero-shot -- run evaluate.py")
        continue
    zero_shot[artist] = hit
    df = hit["df"]
    print(f"{artist}: {df[artist].mean():.4f} +/- {df[artist].std():.4f}")

Gojira: 0.0076 +/- 0.0061
Tool: 0.9571 +/- 0.0364
Death: 0.0132 +/- 0.0176
Meshuggah: 0.0080 +/- 0.0037
Opeth: 0.0024 +/- 0.0013


## B2 -- Few-shot (3 examples)

Artist name + 3 training lyrics as in-context examples.

In [3]:
few_shot = {}
for artist in ARTISTS:
    hit = load_baseline(artist, "few_shot")
    if hit is None:
        print(f"missing {artist} few-shot -- run evaluate.py")
        continue
    few_shot[artist] = hit
    df = hit["df"]
    print(f"{artist}: {df[artist].mean():.4f} +/- {df[artist].std():.4f}")

Gojira: 0.1412 +/- 0.2999
Tool: 0.8842 +/- 0.1902
Death: 0.0109 +/- 0.0075
Meshuggah: 0.0170 +/- 0.0228
Opeth: 0.0035 +/- 0.0019


## Summary

In [4]:
print("Target-artist mean attribution (higher = better):\n")
print(f"{'Artist':<12} {'Zero-shot':>10} {'Few-shot':>10}")
print("-" * 34)
for artist in ARTISTS:
    zs = zero_shot[artist]["df"][artist].mean() if artist in zero_shot else float("nan")
    fs = few_shot[artist]["df"][artist].mean() if artist in few_shot else float("nan")
    print(f"{artist:<12} {zs:>10.4f} {fs:>10.4f}")

Target-artist mean attribution (higher = better):

Artist        Zero-shot   Few-shot
----------------------------------
Gojira           0.0076     0.1412
Tool             0.9571     0.8842
Death            0.0132     0.0109
Meshuggah        0.0080     0.0170
Opeth            0.0024     0.0035
